# Project name - FMCG Sales Drivers Analysis

## Causal and Statistical Analysis of Sales Drivers in FMCG Retail: A Multivariate Study

# 1. Problem Definition (Scientific Method)

## Research Question
    What affects sales?
    Which operational, environmental, and store-level factors significantly drive FMCG beverage sales?

Why it matters (business + analytics)

## Hypotheses
    H1: Store visits positively affect sales volume
    H2: Weather conditions (temperature, precipitation) significantly influence demand
    H3: Store execution (coolers, visibility, equipment) increases sales performance
    H4: Effects differ across store segments and regions

# 2. DATA ARCHITECTURE
## CORE DATA (Business Layer)
### 1. SALES (target dataset)
    date
    customer
    revenue_bgn
    cases
## OPERATIONS LAYER
### 2. VISITS
    Customer
    date
    visits   
### 3. COOLER / EQUIPMENT
    customer
    equipment count
    branding
    status
## MASTER STORE DATA
### 4. LIST DATASET
    customer
    region
    city
    channel
    segment
## EXTERNAL ENVIRONMENT
### 5. WEATHER
    date
    temperature
    precipitation
    humidity
    wind
    holiday
    weekend

    
internal sales system

weather dataset (external source)

holidays dataset

DATA LOADING + DATA QUALITY CHECK

In [1]:
import pandas as pd

# =========================
# LOAD DATASETS
# =========================

sales = pd.read_csv("data/sales_dataset_clean.csv.gz")
visits = pd.read_csv("data/visits_dataset.csv")
cooler = pd.read_csv("data/cooler_dataset.csv")
store = pd.read_csv("data/list_dataset.csv")
weather = pd.read_csv("data/weather_dataset_bg.csv")

# =========================
# CLEAN COLUMN NAMES
# =========================
for df in [sales, visits, cooler, store, weather]:
    df.columns = df.columns.str.strip()

# =========================
# QUICK CHECK (DATA QUALITY)
# =========================

def check(df, name):
    print("\n" + "="*40)
    print(name)
    print("="*40)
    print("Shape:", df.shape)
    print("Missing values:\n", df.isna().sum())
    print("Columns:\n", df.columns.tolist())

check(sales, "SALES")
check(visits, "VISITS")
check(cooler, "COOLER")
check(store, "STORE")
check(weather, "WEATHER")


SALES
Shape: (662041, 4)
Missing values:
 date           0
customer       0
revenue_bgn    0
cases          0
dtype: int64
Columns:
 ['date', 'customer', 'revenue_bgn', 'cases']

VISITS
Shape: (109642, 3)
Missing values:
 Customer                0
Calendar day            0
Executed\nVisits All    0
dtype: int64
Columns:
 ['Customer', 'Calendar day', 'Executed\nVisits All']

COOLER
Shape: (13557, 6)
Missing values:
 Customer                  0
Type                      0
Branding                  0
Status                    0
Number of \nEquipments    0
Number of doors           0
dtype: int64
Columns:
 ['Customer', 'Type', 'Branding', 'Status', 'Number of \nEquipments', 'Number of doors']

STORE
Shape: (12110, 8)
Missing values:
 Customer         0
Customer Name    0
Region           0
City             0
Trade Channel    0
Segment          0
Seasonal         0
Тype             0
dtype: int64
Columns:
 ['Customer', 'Customer Name', 'Region', 'City', 'Trade Channel', 'Segment', 'Seasona

DATA INTEGRATION

In [2]:
# =========================
# FIX VISITS COLUMN NAMES
# =========================

visits.columns = visits.columns.str.strip()

visits = visits.rename(columns={
    "Calendar day": "date",
    "Customer": "customer",
    "Executed\nVisits All": "visits"
})

In [3]:
# =========================
# FIX DATE TYPES
# =========================

sales["date"] = pd.to_datetime(sales["date"])
visits["date"] = pd.to_datetime(visits["date"])
weather["date"] = pd.to_datetime(weather["date"])

# fix visits column name if needed
visits = visits.rename(columns={"Customer": "customer"})

# =========================
# STEP 1: SALES + VISITS
# =========================
df = sales.merge(visits, on=["customer", "date"], how="left")

# =========================
# STEP 2: ADD STORE INFO
# =========================
df = df.merge(store, on="customer", how="left")

# =========================
# STEP 3: ADD COOLER DATA
# =========================
df = df.merge(cooler, on="customer", how="left")

# =========================
# STEP 4: ADD WEATHER
# =========================
df = df.merge(weather, on="date", how="left")

print(df.head())
print(df.shape)

C:\Users\radio\AppData\Local\Temp\ipykernel_4860\3972662974.py:6: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  visits["date"] = pd.to_datetime(visits["date"])


KeyError: 'customer'

 FEATURE ENGINEERING

In [ ]:
# =========================
# FEATURE ENGINEERING
# =========================

df["sales_per_case"] = df["revenue_bgn"] / (df["cases"] + 1)

df["sales_per_visit"] = df["revenue_bgn"] / (df["visits"] + 1)

df["high_visit"] = df["visits"] > df["visits"].median()

df["temperature_bucket"] = pd.cut(
    df["temperature_celsius"],
    bins=[-10, 10, 25, 40],
    labels=["cold", "mild", "hot"]
)

df["is_active_store"] = df["status"].notna()

EDA

In [ ]:
import matplotlib.pyplot as plt

# SALES DISTRIBUTION
df["revenue_bgn"].hist()
plt.title("Sales Distribution")
plt.show()

# VISITS vs SALES
plt.scatter(df["visits"], df["revenue_bgn"])
plt.title("Visits vs Sales")
plt.show()

# TEMPERATURE vs SALES
plt.scatter(df["temperature_celsius"], df["revenue_bgn"])
plt.title("Temperature vs Sales")
plt.show()

3. Data Structure Overview

4. Loading Data

5. Project Plan 

Data cleaning

EDA

Hypothesis testing

Regression modeling

Conclusions